In [ ]:
# Install required packages for running the model on Google Colab
!pip install -q transformers accelerate peft bitsandbytes flask pyngrok

In [ ]:
# Mount Google Drive to access model files
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Verify the contents of the model directory
!ls /content/drive/MyDrive/WikiHow_Project/lora_adapter

In [ ]:
# Load necessary libraries
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

print("Loading models...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# MODEL 1: LLaMA 3B for GENERATION
print("Loading LLaMA 3B for generation...")
gen_model_name = "unsloth/Llama-3.2-3B-Instruct"
gen_tokenizer = AutoTokenizer.from_pretrained(gen_model_name)
gen_model = AutoModelForCausalLM.from_pretrained(
    gen_model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
gen_tokenizer.pad_token = gen_tokenizer.eos_token
gen_model.eval()
print("Generation model loaded!")

# MODEL 2: My Trained LLaMA 1B for CLASSIFICATION
print("Loading my trained model for classification...")
class_model_name = "unsloth/Llama-3.2-1B"
class_tokenizer = AutoTokenizer.from_pretrained(class_model_name)
class_model = AutoModelForCausalLM.from_pretrained(
    class_model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
class_tokenizer.pad_token = class_tokenizer.eos_token

# Loading my fine-tuned LoRA adapter
adapter_path = "/content/drive/MyDrive/WikiHow_Project/lora_adapter"
class_model = PeftModel.from_pretrained(class_model, adapter_path)
class_model.eval()
print("My classification model loaded!")

print(f"\nTotal GPU Memory: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

In [ ]:
# Functions for generation and classification
import re

def generate_instructions(query):
    """Use LLaMA 3B to generate instructions"""
    prompt = f"""Generate clear step-by-step instructions for: {query}

Requirements:
- Provide 6-10 numbered steps
- Each step should be one specific action
- Start directly with step 1, no introduction
- Be practical and detailed

1."""

    messages = [{"role": "user", "content": prompt}]
    input_text = gen_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = gen_tokenizer(input_text, return_tensors="pt").to(gen_model.device)

    with torch.no_grad():
        outputs = gen_model.generate(
            **inputs,
            max_new_tokens=500,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            pad_token_id=gen_tokenizer.eos_token_id
        )

    response = gen_tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = response.replace("assistant", "").replace("Assistant", "")

    steps = []

    # Find numbered steps
    patterns = [
        r'\d+\.\s*\*?\*?([^*\n]+)',
        r'\d+\)\s*([^\n]+)',
    ]

    for pattern in patterns:
        matches = re.findall(pattern, response)
        if matches:
            for text in matches:
                text = text.strip().rstrip('*').strip()
                # Skip intro sentences, empty, or too short
                if len(text) > 10 and len(text) < 500:
                    # Skip if it's an introductory phrase
                    intro_words = ["here's", "here is", "guide", "follow these", "steps to", "instructions for"]
                    if not any(intro in text.lower() for intro in intro_words):
                        steps.append(text)
            break

    # Remove duplicates
    seen = set()
    unique_steps = []
    for step in steps:
        key = step.lower()[:50]
        if key not in seen:
            seen.add(key)
            unique_steps.append(step)

    return unique_steps[:10] if unique_steps else [f"Follow instructions for {query}"]


def classify_instruction(instruction):
    """Use YOUR trained LLaMA 1B to classify an instruction"""

    prompt = f"""### Instruction:
Classify this instruction into one of these categories: Simple, Mandatory, Sequential, Conditional, Exclusive, Goal-based

Instruction: {instruction}

### Response:
"""

    inputs = class_tokenizer(prompt, return_tensors="pt").to(class_model.device)

    with torch.no_grad():
        outputs = class_model.generate(
            **inputs,
            max_new_tokens=30,
            temperature=0.1,
            do_sample=False,
            pad_token_id=class_tokenizer.eos_token_id
        )

    response = class_tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract model output after "### Response:"
    if "### Response:" in response:
        model_output = response.split("### Response:")[-1].strip()
        print(f" Model output: '{model_output[:100]}'")
    else:
        model_output = response

    # Match to valid categories
    categories = ["Sequential", "Conditional", "Exclusive", "Goal-based", "Mandatory", "Simple"]

    for cat in categories:
        if cat.lower() in model_output.lower():
            return cat

    # Smart fallback based on instruction keywords
    instruction_lower = instruction.lower()

    # Sequential: order-based words
    if any(word in instruction_lower for word in ["first", "then", "next", "after", "before", "finally", "lastly", "start by", "begin"]):
        return "Sequential"

    # Conditional: if/when conditions
    if any(word in instruction_lower for word in ["if", "when", "unless", "in case", "depending", "should"]):
        return "Conditional"

    # Mandatory: requirement words
    if any(word in instruction_lower for word in ["must", "required", "always", "never", "essential", "important", "necessary"]):
        return "Mandatory"

    # Exclusive: choice words
    if any(word in instruction_lower for word in ["either", "or", "choose", "alternative", "option"]):
        return "Exclusive"

    # Goal-based: outcome words
    if any(word in instruction_lower for word in ["goal", "achieve", "aim", "result", "ensure", "make sure"]):
        return "Goal-based"

    return "Simple"


def generate_and_classify(query):
    """Generate instructions and classify each one"""

    print(f"Generating instructions for: {query}")

    steps = generate_instructions(query)
    print(f"Generated {len(steps)} steps")

    results = []
    for i, step in enumerate(steps):
        print(f"  Step {i+1}: {step[:60]}...")
        category = classify_instruction(step)
        print(f"    → Category: {category}")
        results.append({
            "instruction": step,
            "category": category
        })

    return results

In [ ]:
# My ngrok authtoken
!ngrok config add-authtoken 38n6Neo7bQjlCr8Q2sD45SC5SGh_2cHztigNHagVZL1inan # Replace with your own authtoken

In [ ]:
# Flask server with ngrok tunneling
from flask import Flask, request, jsonify
from pyngrok import ngrok

app = Flask(__name__)

@app.route('/parse', methods=['POST', 'OPTIONS'])
def parse():
    if request.method == 'OPTIONS':
        response = jsonify({'status': 'ok'})
        response.headers.add('Access-Control-Allow-Origin', '*')
        response.headers.add('Access-Control-Allow-Headers', 'Content-Type')
        response.headers.add('Access-Control-Allow-Methods', 'POST')
        return response

    try:
        data = request.json
        query = data.get('query', data.get('text', ''))

        if not query:
            return jsonify({'error': 'No query provided'}), 400

        instructions = generate_and_classify(query)

        result = {
            'query': query,
            'instructions': instructions
        }

        response = jsonify({'result': result})
        response.headers.add('Access-Control-Allow-Origin', '*')
        return response

    except Exception as e:
        import traceback
        traceback.print_exc()
        return jsonify({'error': str(e)}), 500

@app.route('/health', methods=['GET'])
def health():
    response = jsonify({'status': 'ok', 'model': 'LLaMA-3B + Your Trained Model'})
    response.headers.add('Access-Control-Allow-Origin', '*')
    return response

public_url = ngrok.connect(5000)
print("\n" + "="*60)
print("SERVER RUNNING!")
print("="*60)
print(f"\nURL: {public_url.public_url}\n")
print("="*60)

app.run(port=5000)